# 04 – Huấn luyện mô hình (Model)
Tinh chỉnh **cả 3 thuật toán bằng cùng một tiêu chí** (F1 lớp spam, CV phân tầng 5-fold trên train) để so sánh công bằng.

In [ ]:
import sys
sys.path.insert(0, "..")  # để import được package spam_filter
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
pd.set_option("display.max_colwidth", 110)
pd.set_option("display.width", 200)

from spam_filter.config import DataConfig
from spam_filter.data import load_dataset, split_data

cfg = DataConfig()
df = load_dataset(cfg)
X_train, X_test, y_train, y_test = split_data(df, cfg)

In [ ]:
from spam_filter.pipeline import ExperimentSpec
from spam_filter.runner import run_specs

SHOW = ["id", "cv_f1_train", "precision", "recall", "f1"]

def compare(specs):
    """Chạy các spec, trả về bảng gọn (CV trên train + test). Chọn theo cv_f1_train."""
    res, _ = run_specs(specs, X_train, y_train, X_test, y_test, cfg.cv_folds)
    return res[SHOW].round(4).set_index("id")

In [ ]:
CLEAN = {"replace_url": True, "replace_money": True, "replace_number": True, "remove_punct": True}
VEC = {"kind": "tfidf", "ngram_range": (1, 2), "min_df": 2}

def tune(algo, name, values, key, extra=None):
    specs = [ExperimentSpec(id=f"{algo}:{name}={v}", description=f"{algo} {name}={v}", model=algo,
                            model_params={**(extra or {}), key: v}, cleaner=CLEAN, vectorizer=VEC) for v in values]
    return compare(specs)

print("Cố định: TextCleaner(URL/tiền/số/dấu câu) + TF-IDF", VEC)

## 1. Multinomial Naive Bayes – `alpha`

In [ ]:
nb_res = tune("nb", "alpha", [0.01, 0.05, 0.1, 0.3, 0.5, 1.0], "alpha"); nb_res

## 2. K-NN – số láng giềng `k` (cosine)

In [ ]:
knn_res = tune("knn", "k", [1, 3, 5, 7, 11, 15], "n_neighbors"); knn_res

## 3. Decision Tree – `max_depth`

In [ ]:
tree_res = tune("tree", "max_depth", [5, 10, 20, 40, None], "max_depth", {"min_samples_leaf": 2}); tree_res

## 4. Tổng hợp: cấu hình tốt nhất của mỗi thuật toán (chọn theo `cv_f1_train`)

In [ ]:
best = pd.DataFrame({n: r.sort_values("cv_f1_train", ascending=False).iloc[0]
                     for n, r in {"NaiveBayes": nb_res, "K-NN": knn_res, "DecisionTree": tree_res}.items()}).T
best

## 5. Tham khảo: GridSearchCV cho Naive Bayes (tìm chung tiền xử lý vector và `alpha`)

In [ ]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from spam_filter.preprocessing import TextCleaner

pipe = Pipeline([("clean", TextCleaner(**CLEAN)), ("vec", TfidfVectorizer(ngram_range=(1, 2))), ("clf", MultinomialNB())])
grid = {"vec__min_df": [1, 2, 3], "vec__max_df": [0.9, 1.0], "clf__alpha": [0.01, 0.05, 0.1, 0.3, 0.5, 1.0]}
gs = GridSearchCV(pipe, grid, scoring="f1", n_jobs=1,
                  cv=StratifiedKFold(5, shuffle=True, random_state=cfg.random_state)).fit(X_train, y_train)
print("Tham số tốt nhất:", gs.best_params_, "| CV F1 =", round(gs.best_score_, 4))

Phạm vi tìm & tiêu chí: xem tiêu đề từng mục. Cột `precision/recall/f1` (test) chỉ để tham khảo – **chọn theo `cv_f1_train`** để không chọn thiên lệch theo tập test.

## Nhận xét (nhóm điền)
Vì sao Naive Bayes hợp văn bản dạng đếm? K-NN/Tree yếu hơn ở đâu (chiều cao, thưa)? Giả định nào của từng thuật toán bị vi phạm?